# 导入工具包

In [1]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm
import seedir as sd

# 1.指定数据集路径并查看结构

In [2]:
# 指定你的数据集文件夹名称
Dataset_Path = 'Watermelon87_Semantic_Seg_Labelme'

print("📁 当前数据集目录结构：")
sd.seedir(Dataset_Path, style='emoji', depthlimit=1)

📁 当前数据集目录结构：
📁 Watermelon87_Semantic_Seg_Labelme/
├─📁 ann_dir/
├─📁 img_dir/
├─📁 labelme_jsons/
├─📁 train/
└─📁 val/


# 2.删除系统自动生成的多余文件
使用跨平台的 Python 原生库清理隐藏缓存文件，防止干扰后续的文件划分。

In [3]:
junk_names = ['__MACOSX', '.DS_Store', '.ipynb_checkpoints']

# 💡 这里的关键修复：必须用 Path() 把字符串包装成路径对象
base_path = Path(Dataset_Path)

print("🗑️ 开始检查并清理多余文件...")
deleted_count = 0

for name in junk_names:
    for item in base_path.rglob(name):
        if item.exists():
            try:
                if item.is_dir():
                    shutil.rmtree(item)
                else:
                    item.unlink()
                print(f" ✅ 已删除: {item}")
                deleted_count += 1
            except Exception as e:
                print(f" ❌ 删除失败 {item}: {e}")

if deleted_count == 0:
    print(" ✨ 未发现多余文件，目录很干净！")
else:
    print(f" ✨ 清理完成！共删除了 {deleted_count} 个多余文件/文件夹。")

🗑️ 开始检查并清理多余文件...
 ✨ 未发现多余文件，目录很干净！


# 3. 获取所有图像并打乱顺序，计算划分数量
设置随机数种子以保证每次划分的结果可复现。

In [ ]:
test_frac = 0.2  # 测试集比例
random.seed(123) # 随机数种子，便于复现

img_dir_path = os.path.join(Dataset_Path, 'img_dir')
ann_dir_path = os.path.join(Dataset_Path, 'ann_dir')

# 获取所有的图像文件名
img_paths = [f for f in os.listdir(img_dir_path) if os.path.isfile(os.path.join(img_dir_path, f))]
random.shuffle(img_paths) # 随机打乱

val_number = int(len(img_paths) * test_frac) # 测试集文件个数
train_files = img_paths[val_number:]         # 训练集文件名列表
val_files = img_paths[:val_number]           # 测试集文件名列表

print('数据集划分概览：')
print(f' - 数据集文件总数: {len(img_paths)}')
print(f' - 训练集文件个数: {len(train_files)}')
print(f' - 测试集文件个数: {len(val_files)}')

📊 数据集划分概览：
 - 数据集文件总数: 87
 - 训练集文件个数: 70
 - 测试集文件个数: 17


# 4. 创建 train 和 val 文件夹
直接在 `img_dir` 和 `ann_dir` 内部创建 `train` 和 `val` 文件夹

In [5]:
# 为 img_dir 创建子文件夹
os.makedirs(os.path.join(img_dir_path, 'train'), exist_ok=True)
os.makedirs(os.path.join(img_dir_path, 'val'), exist_ok=True)

# 为 ann_dir 创建子文件夹
os.makedirs(os.path.join(ann_dir_path, 'train'), exist_ok=True)
os.makedirs(os.path.join(ann_dir_path, 'val'), exist_ok=True)

print("✅ 'train' 和 'val' 文件夹创建完成。")

✅ 'train' 和 'val' 文件夹创建完成。


# 5.移动图像和标签文件到对应的文件夹
根据前面划分好的列表，同步移动原图和对应的 Mask 标注图。

In [6]:
print("🚀 开始移动训练集文件 (Training Set)...")
for each in tqdm(train_files):
    # 移动图像
    src_img = os.path.join(img_dir_path, each)
    dst_img = os.path.join(img_dir_path, 'train', each)
    shutil.move(src_img, dst_img)
    
    # 移动对应的标注图 (假设标注图为 .png 格式且文件名主体与图像一致)
    mask_name = each.split('.')[0] + '.png'
    src_mask = os.path.join(ann_dir_path, mask_name)
    dst_mask = os.path.join(ann_dir_path, 'train', mask_name)
    if os.path.exists(src_mask):
        shutil.move(src_mask, dst_mask)

print("🚀 开始移动测试集文件 (Validation Set)...")
for each in tqdm(val_files):
    # 移动图像
    src_img = os.path.join(img_dir_path, each)
    dst_img = os.path.join(img_dir_path, 'val', each)
    shutil.move(src_img, dst_img)
    
    # 移动对应的标注图
    mask_name = each.split('.')[0] + '.png'
    src_mask = os.path.join(ann_dir_path, mask_name)
    dst_mask = os.path.join(ann_dir_path, 'val', mask_name)
    if os.path.exists(src_mask):
        shutil.move(src_mask, dst_mask)
        
print("✨ 所有文件移动完毕！")

🚀 开始移动训练集文件 (Training Set)...


100%|██████████| 70/70 [00:00<00:00, 749.98it/s]


🚀 开始移动测试集文件 (Validation Set)...


100%|██████████| 17/17 [00:00<00:00, 3956.23it/s]

✨ 所有文件移动完毕！


# 6. 检查最终的数据集目录结构

In [7]:
print("📁 最终的数据集目录结构：")
sd.seedir(Dataset_Path, style='emoji', depthlimit=2)

📁 最终的数据集目录结构：
📁 Watermelon87_Semantic_Seg_Labelme/
├─📁 ann_dir/
│ ├─📁 train/
│ └─📁 val/
├─📁 img_dir/
│ ├─📁 train/
│ └─📁 val/
├─📁 labelme_jsons/
│ ├─📄 01bd15599c606aa801201794e1fa30.jpg@1280w_1l_2o_100sh.json
│ ├─📄 045_sozai_l.json
│ ├─📄 04_35-2.json
│ ├─📄 0778_02.json
│ ├─📄 11-loai-trai-cay-dinh-duong-nen-an-hang-ngay-dua-hau.json
│ ├─📄 1471253631_2.json
│ ├─📄 1510702494080_melon_NWS_block-low.json
│ ├─📄 17897490_93d9666602_z.json
│ ├─📄 20170613134012_430390.json
│ ├─📄 202007220195_top_img_A.json
│ ├─📄 21746.1.json
│ ├─📄 2401-food-tip-image-MAIN.json
│ ├─📄 24a3a0f8a11ae29f8ccca35c822ac0e2991b4773.json
│ ├─📄 31cc9b997d1e3a8ed5b6a3cbf5efd86d.json
│ ├─📄 360_F_85084369_iab0VH1ohx2lHtM3OfCncfE9I7VPpl0N.json
│ ├─📄 468D11D9CB53158C55CF126CFE3D49C0A070758A_size321_w400_h400.json
│ ├─📄 5b3c8018N634d43bd.json
│ ├─📄 68200_main-1.json
│ ├─📄 69735779_m-5b756b48997fa.json
│ ├─📄 7c75f4c6f3bdd9d61d8782ddff1ff6ac.json
│ ├─📄 8-thuc-pham-chong-cam-cum-1459496975020.json
│ ├─📄 8fe7f149146b6866.json
│ ├─📄 